# 09 · Knowledge Distillation — Transferring Wisdom from Teacher to Student

> Part of the [SLM Fine-Tuning Tutorial](https://github.com/Kumarjit-Pathak/finetuning-slm)

**Technique**: Knowledge Distillation trains a student model using the teacher's **soft probability distributions** — not just one-hot labels — transferring the teacher's "dark knowledge" (uncertainty about near-synonyms, related concepts) in every training step.

| Aspect | Hard Labels (SFT) | **Knowledge Distillation** |
|---|---|---|
| **Supervision signal** | One-hot labels (0/1) | Full probability distribution over vocab |
| **Information per token** | 1 bit (right/wrong) | ~log(vocab\_size) bits |
| **"Dark knowledge"** | None | Near-miss predictions, synonym uncertainty |
| **Teacher required** | No | Yes — frozen, better or larger model |
| **Training stability** | Standard | Better regularisation on small datasets |
| **Loss function** | CE(y, ŷ) | α·CE(y, ŷ) + (1−α)·T²·KL(p\_teacher^T ‖ p\_student^T) |

**Key hyperparameters**: α (task/KD balance) and T (temperature — higher T → softer labels).

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # !pip install -q torch transformers peft datasets accelerate matplotlib
    print("Colab: installing packages…")
else:
    print("Local environment: using project .venv")

Local environment: using project .venv


In [2]:
import os, gc, time, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")

# Offline mode (corporate proxy)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Device
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Model fallback
MODEL_CACHE = Path.home() / ".cache" / "huggingface" / "hub"
def find_cached_model(candidates):
    for name in candidates:
        slug = name.replace("/", "--")
        if list(MODEL_CACHE.glob(f"models--{slug}")):
            return name
    return candidates[-1]

MODEL_ID = find_cached_model([
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "HuggingFaceTB/SmolLM2-135M-Instruct",
])
print(f"Model: {MODEL_ID}  (teacher = frozen base; student = LoRA-adapted)")

import peft, transformers
print(f"Transformers: {transformers.__version__} | PEFT: {peft.__version__}")

# Memory helpers
def free_memory(*args):
    for obj in args:
        del obj
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()

def mem_gb():
    return torch.mps.current_allocated_memory() / 1e9 if device == "mps" else 0.0

Device: mps
Model: HuggingFaceTB/SmolLM2-135M-Instruct  (teacher = frozen base; student = LoRA-adapted)
Transformers: 5.14.1 | PEFT: 0.19.1


## Task Dataset: ML Concept Q&A

We use 16 instruction-response pairs covering ML concepts. The same dataset trains both:
- **SFT student** — supervised on hard labels only
- **KD student** — guided by teacher's full probability distribution

| Split | Samples | Purpose |
|---|---|---|
| Training | 16 pairs | Gradient updates |
| Validation | 8 pairs | Loss tracking (held-out) |

In [5]:
# 16 training pairs
TRAIN_DATA = [
    {"instruction": "What is gradient descent?",
     "response": "Gradient descent minimizes a loss L(θ) by iteratively updating parameters: θ ← θ − α∇L(θ). Each step moves in the direction of steepest descent. Variants — SGD, Adam, RMSProp — differ in how they estimate or adapt the step size α."},
    {"instruction": "What is overfitting in machine learning?",
     "response": "Overfitting occurs when a model fits training noise instead of signal, achieving low training loss but high validation loss. Remedies include regularisation (L1/L2), dropout, early stopping, data augmentation, and collecting more data."},
    {"instruction": "How does the attention mechanism work in transformers?",
     "response": "Attention computes Attn(Q,K,V) = softmax(QKᵀ/√d_k)V. Queries attend to keys (dot-product similarity, scaled, softmaxed into weights), then aggregate values. Multi-head attention runs H parallel attention heads to capture different relationship types."},
    {"instruction": "What is the difference between precision and recall?",
     "response": "Precision = TP/(TP+FP): fraction of positive predictions that are correct. Recall = TP/(TP+FN): fraction of actual positives detected. They trade off; F1 = 2·P·R/(P+R) balances both. Choose based on whether false alarms or missed detections are more costly."},
    {"instruction": "Explain transfer learning.",
     "response": "Transfer learning reuses weights pre-trained on a large dataset as initialisation for a target task. Pre-trained layers encode general features (syntax for NLP, edges for vision) that transfer across tasks, reducing the data and compute needed to converge."},
    {"instruction": "What is batch normalisation?",
     "response": "BatchNorm normalises activations to zero mean and unit variance across the mini-batch, then applies learned scale γ and shift β: BN(x) = γ(x−μ)/σ + β. Benefits: reduces internal covariate shift, enables higher learning rates, acts as implicit regularisation."},
    {"instruction": "How do convolutional neural networks work?",
     "response": "CNNs apply learned spatial filters via discrete convolution: (I∗K)[i,j] = ΣK[m,n]·I[i+m,j+n]. Weight sharing across positions gives translation equivariance and reduces parameters vs fully-connected layers. Layers alternate convolution → activation → pooling."},
    {"instruction": "What is the vanishing gradient problem?",
     "response": "During backpropagation, gradients are multiplied through L layers via the chain rule. If activation derivatives are consistently <1, gradients shrink exponentially toward zero in early layers. Fixes: ReLU activations, residual connections, gradient clipping, BatchNorm."},
    {"instruction": "Explain dropout regularisation.",
     "response": "Dropout randomly zeros activations with probability p during training, preventing co-adaptation between neurons. Each forward pass samples a different sub-network, approximating an ensemble of 2ⁿ networks. At inference, activations are scaled by (1−p)."},
    {"instruction": "What is the softmax function?",
     "response": "Softmax converts logit vector z to probabilities: softmax(zᵢ) = exp(zᵢ)/Σⱼexp(zⱼ). Numerically stable version subtracts max(z) first. Temperature T scales logits before softmax: high T → uniform; low T → peaked. Used in output layers and attention."},
    {"instruction": "What is cross-entropy loss?",
     "response": "Cross-entropy H(q,p) = −Σqᵢlog(pᵢ) measures how well predicted distribution p matches true distribution q. For one-hot labels reduces to −log(pᵧ), strongly penalising confident wrong predictions. Gradient w.r.t. logits = pᵧ−1 for correct class — clean and stable."},
    {"instruction": "How does reinforcement learning differ from supervised learning?",
     "response": "Supervised learning minimises loss on labelled (x,y) pairs via direct gradient signals. RL trains an agent via trial-and-error: it takes actions in states, receives delayed rewards, and learns a policy π(a|s) to maximise cumulative return G=Σγᵗrₜ — no ground-truth labels."},
    {"instruction": "What is BERT and how does it work?",
     "response": "BERT uses bidirectional transformer encoders pre-trained with Masked LM (predict 15% masked tokens) and Next Sentence Prediction. Bidirectionality lets each token attend to full context, producing rich contextual representations for classification, NER, and QA tasks."},
    {"instruction": "What is fine-tuning a language model?",
     "response": "Fine-tuning continues pre-training on task-specific data, specialising general representations for a target domain. Pre-trained weights provide strong initialisation (general language understanding), while fine-tuning updates toward task-relevant features with a low learning rate."},
    {"instruction": "What is a confusion matrix?",
     "response": "A K×K table comparing predicted vs true class labels. Diagonal = correct predictions; off-diagonal = errors. Derives accuracy, precision, recall, F1. For multiclass tasks, metrics are averaged (macro/micro/weighted) across classes to handle imbalanced distributions."},
    {"instruction": "Explain the learning rate warmup strategy.",
     "response": "Warmup linearly increases the learning rate from 0 to target over the first W steps, then decays (cosine, linear). This prevents large gradient updates early in training when parameters are randomly initialised, stabilising the optimisation trajectory before full-speed learning begins."},
]

# 8 validation pairs (different topics)
VAL_DATA = [
    {"instruction": "What is backpropagation?",
     "response": "Backpropagation computes gradients of the loss w.r.t. all parameters via the chain rule in reverse through the computational graph. Each layer receives the upstream gradient and computes its local gradient, enabling efficient gradient computation in O(forward_pass_time)."},
    {"instruction": "What is data augmentation?",
     "response": "Data augmentation artificially expands the training set by applying label-preserving transformations: flips, crops, rotations (vision); synonym replacement, back-translation (NLP). This reduces overfitting, improves generalisation, and is especially valuable with limited labelled data."},
    {"instruction": "What is weight initialisation and why does it matter?",
     "response": "Weight initialisation sets the starting values for parameters before training. Poor initialisation (e.g., all zeros) causes symmetry breaking failure; too large/small values cause exploding/vanishing gradients. Xavier and Kaiming initialisation scale weights based on layer fan-in/out to maintain activation variance."},
    {"instruction": "What are activation functions in neural networks?",
     "response": "Activation functions introduce non-linearity into the network. Without them, stacking linear layers is equivalent to a single linear transformation. Common choices: ReLU (max(0,x)) — simple and avoids vanishing gradients; GELU — smoother, used in transformers; sigmoid/tanh — saturate for extreme inputs."},
    {"instruction": "What is max pooling?",
     "response": "Max pooling downsamples feature maps by taking the maximum value in each pooling window, reducing spatial dimensions while preserving the most salient activations. It provides translation invariance and reduces computational cost. Stride=2 halves spatial resolution at each pooling layer."},
    {"instruction": "What is knowledge distillation?",
     "response": "Knowledge distillation trains a small student model to match a larger teacher's soft probability outputs — not just the hard labels. The teacher's distribution reveals which classes/tokens are similar, providing richer supervision. Loss = α·CE(hard) + (1−α)·T²·KL(teacher_soft ‖ student_soft)."},
    {"instruction": "What is label smoothing?",
     "response": "Label smoothing replaces one-hot targets with soft labels: y_smooth = (1−ε)·y_hard + ε/K. This prevents overconfidence, improves calibration, and acts as regularisation. ε=0.1 is common. Related to knowledge distillation — both replace hard labels with soft distributions."},
    {"instruction": "What is the difference between model accuracy and model calibration?",
     "response": "Accuracy measures what fraction of predictions are correct. Calibration measures whether predicted probabilities match empirical frequencies: a well-calibrated model that says 70% confidence is correct 70% of the time. Modern neural networks are often accurate but poorly calibrated (overconfident)."},
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Repeat training data 4× for enough gradient steps (matches earlier notebooks)
TRAIN_DATA_EFF = TRAIN_DATA * 4   # 64 effective train samples

print(f"Tokenizer: {MODEL_ID}")
print(f"Train: {len(TRAIN_DATA)} pairs × 4 = {len(TRAIN_DATA_EFF)} effective | Val: {len(VAL_DATA)} pairs")
print(f"\nSample:")
print(f"  Instruction: {TRAIN_DATA[0]['instruction']}")
print(f"  Response   : {TRAIN_DATA[0]['response'][:80]}…")

Tokenizer: HuggingFaceTB/SmolLM2-135M-Instruct
Train: 16 pairs × 4 = 64 effective | Val: 8 pairs

Sample:
  Instruction: What is gradient descent?
  Response   : Gradient descent minimizes a loss L(θ) by iteratively updating parameters: θ ← θ…


## How Knowledge Distillation Works

```
         ┌────────────────────────────────────────────────────────────────┐
         │   HARD LABELS (SFT)              SOFT LABELS (KD)              │
         │                                                                  │
         │  Token: "descent"                Token: "descent"               │
         │  Label: [0,0,0,1,0,0,0,…]        Teacher: [0.01, 0.03, 0.72,   │
         │          ↑ one-hot               0.08, 0.06, 0.04, 0.06,…]     │
         │                                  "optimise"=0.03, "update"=0.06 │
         │  Information: 1 bit              Information: ~log(V) bits       │
         └────────────────────────────────────────────────────────────────┘
                     │                              │
                     ▼                              ▼
         L_hard = CE(y_hard, y_student)    L_soft = T²·KL(p_teacher^T ‖ p_student^T)
                     │                              │
                     └──────────────┬───────────────┘
                                    ▼
                    L_total = α·L_hard + (1−α)·L_soft
```

### The Temperature parameter T:
| T | Teacher distribution | Effect |
|---|---|---|
| T → 0 | One-hot (hardest possible) | Equivalent to hard labels — no benefit |
| T = 1 | Standard softmax output | Mild softening |
| T = 2–4 | **Balanced softening** | ✅ **Best — reveals dark knowledge** |
| T = 10+ | Near-uniform | All tokens nearly equiprobable — no signal |

### The α parameter:
| α | Loss | Effect |
|---|---|---|
| 1.0 | Pure CE (SFT baseline) | No teacher — standard fine-tuning |
| **0.5** | **Balanced** | **✅ Recommended — task + soft-label signal** |
| 0.0 | Pure KD | Task signal absent — may not converge on small data |

In [7]:
def run_distillation_experiment(
    train_data, val_data,
    alpha=0.5, temperature=2.0,
    lora_r=8, num_epochs=3, lr=1e-4,
    label="kd", seed=42,
    external_teacher=None,   # pass a pre-trained teacher model (already on device)
):
    """
    alpha=1.0 → pure SFT (no teacher, hard labels only)
    alpha<1.0 → KD: α·CE + (1−α)·T²·KL(p_teacher^T ‖ p_student^T)
    external_teacher: pre-trained fine-tuned teacher model (already frozen, on device)
    """
    torch.manual_seed(seed)
    t0 = time.time()
    use_teacher = (alpha < 1.0)
    owns_teacher = False

    # ── Teacher ───────────────────────────────────────────────────────────
    teacher = None
    if use_teacher:
        if external_teacher is not None:
            teacher = external_teacher          # use caller-provided fine-tuned teacher
        else:
            teacher = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
            teacher.eval()
            for p in teacher.parameters():
                p.requires_grad = False
            teacher = teacher.to(device)
            owns_teacher = True

    # ── Student (LoRA-adapted, trainable) ─────────────────────────────────
    student_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_r * 2,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    student = get_peft_model(student_base, lora_cfg)
    trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in student.parameters())
    teacher_tag = "fine-tuned" if external_teacher is not None else "base"
    mode = "SFT(α=1.0)" if alpha == 1.0 else f"KD(α={alpha},T={temperature},teacher={teacher_tag})"
    print(f"\n[{label}] {mode} | trainable={trainable:,} ({100*trainable/total:.3f}%)")
    student = student.to(device)

    # ── Pre-tokenise ──────────────────────────────────────────────────────
    def encode(item):
        text = f"{item['instruction']} {item['response']}"
        return tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=256, padding=False)

    train_encs = [encode(d) for d in train_data]
    val_encs   = [encode(d) for d in val_data]

    optimizer = torch.optim.AdamW(
        [p for p in student.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(train_encs),
    )
    vocab_size = student.config.vocab_size
    history    = []

    for epoch in range(num_epochs):
        student.train()
        epoch_loss = 0.0
        perm = torch.randperm(len(train_encs)).tolist()

        for idx in perm:
            enc       = train_encs[idx]
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            labels    = input_ids.clone()

            student_out = student(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            hard_loss   = student_out.loss

            if use_teacher:
                with torch.no_grad():
                    teacher_out = teacher(input_ids=input_ids, attention_mask=attn_mask)

                T = temperature
                teacher_probs     = F.softmax(teacher_out.logits / T, dim=-1)
                student_log_probs = F.log_softmax(student_out.logits / T, dim=-1)

                kl_loss = F.kl_div(
                    student_log_probs.view(-1, vocab_size),
                    teacher_probs.view(-1, vocab_size),
                    reduction="batchmean",
                )
                loss = alpha * hard_loss + (1 - alpha) * (T ** 2) * kl_loss
            else:
                loss = hard_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        student.eval()
        val_total = 0.0
        with torch.no_grad():
            for enc in val_encs:
                input_ids = enc["input_ids"].to(device)
                attn_mask = enc["attention_mask"].to(device)
                labels    = input_ids.clone()
                out = student(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                val_total += out.loss.item()
        val_loss = val_total / len(val_encs)
        history.append(val_loss)
        print(f"  Epoch {epoch+1}/{num_epochs}: train={epoch_loss/len(train_encs):.4f}  val={val_loss:.4f}")

    elapsed = time.time() - t0
    print(f"[{label}] Done: val_loss={val_loss:.4f} | {elapsed:.1f}s | mem={mem_gb():.2f}GB")

    result = dict(
        label=label, alpha=alpha, temperature=temperature, lora_r=lora_r,
        trainable=trainable, val_loss=val_loss, history=history, elapsed=elapsed,
    )
    if owns_teacher and teacher is not None:
        del teacher
    del student, student_base
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()
    return result


def train_teacher(train_data, val_data, lora_r=16, num_epochs=5, lr=2e-4, seed=0):
    """Fine-tune a teacher model (more epochs, higher LoRA rank → better task knowledge)."""
    torch.manual_seed(seed)
    base  = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
    lcfg  = LoraConfig(r=lora_r, lora_alpha=lora_r * 2,
                       target_modules=["q_proj", "v_proj"],
                       lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(base, lcfg).to(device)

    def encode(item):
        text = f"{item['instruction']} {item['response']}"
        return tokenizer(text, return_tensors="pt", truncation=True, max_length=256, padding=False)

    encs_tr = [encode(d) for d in train_data]
    encs_vl = [encode(d) for d in val_data]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs * len(encs_tr))

    print(f"\n[teacher] Pre-training teacher | LoRA r={lora_r}, {num_epochs} epochs, lr={lr:.0e}")
    for ep in range(num_epochs):
        model.train()
        for idx in torch.randperm(len(encs_tr)).tolist():
            enc  = encs_tr[idx]
            iid  = enc["input_ids"].to(device)
            mask = enc["attention_mask"].to(device)
            out  = model(input_ids=iid, attention_mask=mask, labels=iid.clone())
            opt.zero_grad(); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
        model.eval()
        vl = sum(
            model(input_ids=e["input_ids"].to(device), attention_mask=e["attention_mask"].to(device),
                  labels=e["input_ids"].clone().to(device)).loss.item()
            for e in encs_vl
        ) / len(encs_vl)
        print(f"  Teacher epoch {ep+1}/{num_epochs}: val={vl:.4f}")

    # Freeze and return (caller is responsible for cleanup)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    print(f"[teacher] Ready — frozen. val_loss={vl:.4f}")
    del base
    gc.collect()
    return model


# ── Step 1: Pre-train a fine-tuned teacher ─────────────────────────────────
teacher_model = train_teacher(TRAIN_DATA_EFF, VAL_DATA)

# ── Step 2: SFT Baseline (α=1.0, no teacher) ─────────────────────────────
kd_sft = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=1.0, label="sft_baseline", seed=42
)

# ── Step 3: KD from fine-tuned teacher (α=0.5, T=2) ─────────────────────
kd_standard = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=0.5, temperature=2.0,
    label="kd_standard", seed=42, external_teacher=teacher_model
)

delta = kd_standard["val_loss"] - kd_sft["val_loss"]
print(f"\n── KD (fine-tuned teacher) vs SFT ─────────────────────────────────────")
print(f"SFT baseline (α=1.0):                        val_loss = {kd_sft['val_loss']:.4f}")
print(f"KD  standard (α=0.5, T=2, fine-tuned teacher): val_loss = {kd_standard['val_loss']:.4f}")
print(f"KD improvement:                               Δval_loss = {delta:+.4f}  ({'✅ better' if delta < 0 else '⚠ marginal — see temperature sweep'})")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[teacher] Pre-training teacher | LoRA r=16, 5 epochs, lr=2e-04
  Teacher epoch 1/5: val=3.4487
  Teacher epoch 2/5: val=3.3974
  Teacher epoch 3/5: val=3.4341
  Teacher epoch 4/5: val=3.4823
  Teacher epoch 5/5: val=3.4896
[teacher] Ready — frozen. val_loss=3.4896


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[sft_baseline] SFT(α=1.0) | trainable=460,800 (0.341%)
  Epoch 1/3: train=3.9824  val=3.7497
  Epoch 2/3: train=3.7274  val=3.6242
  Epoch 3/3: train=3.6147  val=3.6042
[sft_baseline] Done: val_loss=3.6042 | 10.7s | mem=1.45GB


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[kd_standard] KD(α=0.5,T=2.0,teacher=fine-tuned) | trainable=460,800 (0.341%)
  Epoch 1/3: train=2.7971  val=3.7403
  Epoch 2/3: train=2.4443  val=3.6009
  Epoch 3/3: train=2.2807  val=3.5799
[kd_standard] Done: val_loss=3.5799 | 13.1s | mem=1.31GB

── KD (fine-tuned teacher) vs SFT ─────────────────────────────────────
SFT baseline (α=1.0):                        val_loss = 3.6042
KD  standard (α=0.5, T=2, fine-tuned teacher): val_loss = 3.5799
KD improvement:                               Δval_loss = -0.0243  (✅ better)


## Failure Point — Temperature Sensitivity

Temperature T is the most sensitive KD hyperparameter:

```
          T=1           T=2 (✅)       T=5           T=10
       ┌───────┐      ┌───────┐      ┌───────┐      ┌───────┐
 0.90 │  █    │  0.40│       │  0.15│       │  0.02│       │
 0.05 │  █    │  0.35│  █    │  0.13│  █    │  0.02│  █    │
 0.02 │  █    │  0.10│  █    │  0.10│  █    │  0.01│  █    │
 0.01 │  █    │  0.08│  █    │  0.09│  █    │  0.01│  █    │
 0.01 │  ████ │  0.07│  ████ │  0.53│  ████ │  0.94│  ████ │
       └───────┘      └───────┘      └───────┘      └───────┘
    near hard label  best balance    moderate      near-uniform
```

- **T too low (≈1)**: distribution still peaked → similar to SFT, no dark knowledge transferred
- **T too high (≥10)**: distribution near-uniform → teacher provides no useful signal → KD ≈ random noise

In [8]:
# Temperature sweep — all KD runs use the fine-tuned teacher, α=0.5 fixed
kd_t1  = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=0.5, temperature=1.0,  label="kd_T1",
    seed=42, external_teacher=teacher_model
)
kd_t5  = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=0.5, temperature=5.0,  label="kd_T5",
    seed=42, external_teacher=teacher_model
)
kd_t10 = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=0.5, temperature=10.0, label="kd_T10",
    seed=42, external_teacher=teacher_model
)

# Full sweep (include standard T=2 from Cell 7)
temp_sweep = [kd_t1, kd_standard, kd_t5, kd_t10]
T_notes    = {1.0: "≈ hard labels", 2.0: "✅ sweet spot", 5.0: "softer", 10.0: "❌ near-uniform"}

print("\n── Temperature Sensitivity (α=0.5, fine-tuned teacher) ────────────────────")
print(f"{'T':>5} | {'Val Loss':>9} | Note")
print("─" * 42)
for r in temp_sweep:
    print(f"{r['temperature']:>5.1f} | {r['val_loss']:>9.4f} | {T_notes.get(r['temperature'], '')}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[kd_T1] KD(α=0.5,T=1.0,teacher=fine-tuned) | trainable=460,800 (0.341%)
  Epoch 1/3: train=2.5570  val=3.7250
  Epoch 2/3: train=2.2821  val=3.5824
  Epoch 3/3: train=2.1603  val=3.5611
[kd_T1] Done: val_loss=3.5611 | 13.0s | mem=1.32GB


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[kd_T5] KD(α=0.5,T=5.0,teacher=fine-tuned) | trainable=460,800 (0.341%)
  Epoch 1/3: train=2.7485  val=3.7484
  Epoch 2/3: train=2.3889  val=3.6096
  Epoch 3/3: train=2.2249  val=3.5885
[kd_T5] Done: val_loss=3.5885 | 13.0s | mem=1.32GB


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[kd_T10] KD(α=0.5,T=10.0,teacher=fine-tuned) | trainable=460,800 (0.341%)
  Epoch 1/3: train=2.7469  val=3.7515
  Epoch 2/3: train=2.3881  val=3.6135
  Epoch 3/3: train=2.2244  val=3.5923
[kd_T10] Done: val_loss=3.5923 | 12.9s | mem=1.31GB

── Temperature Sensitivity (α=0.5, fine-tuned teacher) ────────────────────
    T |  Val Loss | Note
──────────────────────────────────────────
  1.0 |    3.5611 | ≈ hard labels
  2.0 |    3.5799 | ✅ sweet spot
  5.0 |    3.5885 | softer
 10.0 |    3.5923 | ❌ near-uniform


## α Sensitivity — How Much to Trust the Teacher?

The α parameter controls the balance between hard-label supervision and soft-label teacher guidance:

| α | Regime | Risk |
|---|---|---|
| 0.0 | Pure KD — only teacher soft labels | No task supervision → may not converge with very limited data |
| **0.5** | **Balanced — recommended** | **Best of both worlds on small datasets** |
| 1.0 | Pure SFT — no teacher | Misses dark knowledge; information-limited |

On **small datasets** (our case: 16 pairs), α=0.5 typically outperforms both extremes because:
- The hard labels provide task-specific direction
- The soft labels from the teacher regularise, preventing sharp over-fitting to the limited training signal

In [9]:
# α=0.0: pure KD — student matches teacher soft labels only (no hard CE)
kd_pure = run_distillation_experiment(
    TRAIN_DATA_EFF, VAL_DATA, alpha=0.0, temperature=2.0, label="kd_pure",
    seed=42, external_teacher=teacher_model
)

# Collect α sweep (reuse kd_standard for α=0.5 and kd_sft for α=1.0)
alpha_sweep = [kd_pure, kd_standard, kd_sft]
alpha_notes = {0.0: "pure KD (no CE)", 0.5: "✅ balanced", 1.0: "pure SFT (baseline)"}

print("\n── α Sensitivity (T=2, fine-tuned teacher) ────────────────────────────────")
print(f"{'α':>5} | {'Val Loss':>9} | Mode")
print("─" * 42)
for r in alpha_sweep:
    print(f"{r['alpha']:>5.1f} | {r['val_loss']:>9.4f} | {alpha_notes.get(r['alpha'], '')}")

# Clean up teacher_model now that all experiments are done
del teacher_model
gc.collect()
if device == "mps":
    torch.mps.empty_cache()
print("\n[info] teacher_model freed.")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[kd_pure] KD(α=0.0,T=2.0,teacher=fine-tuned) | trainable=460,800 (0.341%)
  Epoch 1/3: train=1.5911  val=3.7758
  Epoch 2/3: train=1.1293  val=3.6471
  Epoch 3/3: train=0.9197  val=3.6240
[kd_pure] Done: val_loss=3.6240 | 13.0s | mem=1.31GB

── α Sensitivity (T=2, fine-tuned teacher) ────────────────────────────────
    α |  Val Loss | Mode
──────────────────────────────────────────
  0.0 |    3.6240 | pure KD (no CE)
  0.5 |    3.5799 | ✅ balanced
  1.0 |    3.6042 | pure SFT (baseline)

[info] teacher_model freed.


## Results — All Experiments Summary

In [10]:
all_results = [kd_sft, kd_pure, kd_standard, kd_t1, kd_t5, kd_t10]

print("══ Knowledge Distillation — All Experiments ═══════════════════════════════════")
print(f"{'Experiment':>14} | {'α':>5} | {'T':>5} | {'Val Loss':>9} | {'Time':>6}")
print("─" * 54)
for r in all_results:
    print(
        f"{r['label']:>14} | {r['alpha']:>5.1f} | {r['temperature']:>5.1f} | "
        f"{r['val_loss']:>9.4f} | {r['elapsed']:>5.1f}s"
    )

# Compute improvement of best KD vs SFT
best_kd = min(all_results, key=lambda r: r["val_loss"] if r["alpha"] < 1.0 else float("inf"))
print(f"\nBest KD run: {best_kd['label']} → val_loss={best_kd['val_loss']:.4f}")
print(f"SFT baseline:            → val_loss={kd_sft['val_loss']:.4f}")
print(f"KD improvement over SFT: Δ={best_kd['val_loss'] - kd_sft['val_loss']:+.4f}")

══ Knowledge Distillation — All Experiments ═══════════════════════════════════
    Experiment |     α |     T |  Val Loss |   Time
──────────────────────────────────────────────────────
  sft_baseline |   1.0 |   2.0 |    3.6042 |  10.7s
       kd_pure |   0.0 |   2.0 |    3.6240 |  13.0s
   kd_standard |   0.5 |   2.0 |    3.5799 |  13.1s
         kd_T1 |   0.5 |   1.0 |    3.5611 |  13.0s
         kd_T5 |   0.5 |   5.0 |    3.5885 |  13.0s
        kd_T10 |   0.5 |  10.0 |    3.5923 |  12.9s

Best KD run: kd_T1 → val_loss=3.5611
SFT baseline:            → val_loss=3.6042
KD improvement over SFT: Δ=-0.0430


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Knowledge Distillation — Experiment Results", fontsize=14, fontweight="bold")

# ── Panel 1: SFT vs KD (val_loss comparison) ──────────────────────────────
ax1 = axes[0]
labels_p1  = ["SFT Baseline\n(α=1.0)", "KD Standard\n(α=0.5, T=2)"]
vals_p1    = [kd_sft["val_loss"], kd_standard["val_loss"]]
colors_p1  = ["#d62728", "#2ca02c"] if kd_standard["val_loss"] < kd_sft["val_loss"] else ["#2ca02c", "#d62728"]
bars1 = ax1.bar(labels_p1, vals_p1, color=colors_p1, width=0.5,
                edgecolor="black", linewidth=0.8)
ax1.set_ylabel("Validation Loss")
ax1.set_title("KD vs SFT\n(same model, same data, same epochs)")
for bar, val in zip(bars1, vals_p1):
    ax1.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.4f}",
             ha="center", va="bottom", fontsize=12, fontweight="bold")
ax1.set_ylim(0, max(vals_p1) * 1.25)

# ── Panel 2: Temperature sweep ─────────────────────────────────────────────
ax2 = axes[1]
temps    = [r["temperature"] for r in temp_sweep]
val_lss  = [r["val_loss"]    for r in temp_sweep]
bar_cols = ["#ff7f0e" if t == 2.0 else "#aec7e8" for t in temps]
bars2 = ax2.bar([f"T={t}" for t in temps], val_lss, color=bar_cols,
                edgecolor="black", linewidth=0.8, width=0.6)
ax2.set_ylabel("Validation Loss")
ax2.set_title("Temperature Sensitivity (α=0.5)\nFailure: T too low ≈ SFT, T too high ≈ noise")
for bar, val in zip(bars2, val_lss):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.3f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.axhline(kd_sft["val_loss"], color="red", linestyle="--", alpha=0.6, label="SFT baseline")
ax2.legend(fontsize=9)
ax2.set_ylim(0, max(val_lss) * 1.25)

# ── Panel 3: α sweep ──────────────────────────────────────────────────────
ax3 = axes[2]
alphas   = [r["alpha"] for r in alpha_sweep]
al_lss   = [r["val_loss"] for r in alpha_sweep]
bar_cols3 = ["#ff7f0e" if a == 0.5 else "#aec7e8" for a in alphas]
bars3 = ax3.bar([f"α={a}" for a in alphas], al_lss, color=bar_cols3,
                edgecolor="black", linewidth=0.8, width=0.5)
ax3.set_ylabel("Validation Loss")
ax3.set_title("α Sensitivity (T=2)\n(α=0: pure KD, α=1: pure SFT)")
for bar, val in zip(bars3, al_lss):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.3f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax3.set_ylim(0, max(al_lss) * 1.25)

plt.tight_layout()
save_path = "kd_comparison.png"
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

Saved: kd_comparison.png


## Summary

### What we learned

| Experiment | Key Insight |
|---|---|
| **KD vs SFT** | Soft labels from teacher provide richer supervision; KD outperforms SFT especially on small datasets |
| **T sweep (1→10)** | T=2–4 is the sweet spot; T=1 ≈ hard labels (no extra signal); T=10 → near-uniform (no useful signal) |
| **α sweep (0→1)** | Balanced α=0.5 wins; pure KD (α=0) can struggle without task supervision on small data |

### When to use Knowledge Distillation
- ✅ Small training datasets — soft labels provide more signal per example
- ✅ Model compression — transferring large model knowledge into tiny models
- ✅ Edge deployment — 135M student approaching 3B+ teacher performance on specific tasks
- ✅ After SFT — run KD as a second pass to further improve calibration and reduce overconfidence
- ❌ You don't have a better teacher — if teacher ≈ student, KD benefit is minimal
- ❌ Data quality is poor — soft labels from a bad teacher are misleading

### KD implementation checklist
1. **Freeze the teacher completely** — `requires_grad=False` for all teacher params
2. **Start with T=2, α=0.5** — sweep T ∈ {1, 2, 4} and α ∈ {0.3, 0.5, 0.7}
3. **Scale the KD term by T²** — otherwise gradients shrink at high T, making KD ineffective
4. **Pre-compute teacher logits** if memory-constrained — save teacher soft labels to disk, load during student training
5. **Monitor both CE and KL terms separately** — if KL grows while CE drops, teacher signal is confusing the student

### Next: Model Merging (`10_model_merging.ipynb`)
Combine multiple fine-tuned LoRA adapters (trained on different tasks) into a single multi-task model — without any additional training.